<a href="https://colab.research.google.com/github/HuyLuong2002/anfis-breast-cancer/blob/main/anfis_breast_cancer_model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cài đặt thư viện



In [2]:

!pip install -q pandas numpy scikit-learn scipy matplotlib seaborn ucimlrepo

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import mahalanobis
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)  # Đảm bảo kết quả tái lập được

# Nạp dữ liệu & Tiền xử lý (MICE + Loại ngoại lai)

In [8]:
# 1. Tải WBCD từ UCI
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"

cols = [
    'ID', 'Clump_Thickness', 'Uniformity_Cell_Size',
    'Uniformity_Cell_Shape', 'Marginal_Adhesion',
    'Single_Epithelial_Cell_Size', 'Bare_Nuclei',
    'Bland_Chromatin', 'Normal_Nucleoli', 'Mitoses',
    'Class'
]

df = pd.read_csv(url, names=cols, na_values='?')

print("=" * 80)
print("DỮ LIỆU GỐC")
print(df.head())
print("\nKích thước:", df.shape)
print("\nSố giá trị thiếu:")
print(df.isnull().sum())

# Xóa cột ID
df.drop('ID', axis=1, inplace=True)

print("\n" + "=" * 80)
print("TRƯỚC KHI GẮN NHÃN")
print(df[['Class']].head(10))
print("\nPhân bố Class:")
print(df['Class'].value_counts())

# Gắn nhãn
df['Class'] = df['Class'].map({2: 0, 4: 1})

print("\n" + "=" * 80)
print("SAU KHI GẮN NHÃN")
print(df[['Class']].head(10))
print("\nPhân bố Class:")
print(df['Class'].value_counts())

feature_names = df.columns[:-1].tolist()
X = df[feature_names]
y = df['Class']

# ------------------------------------------------------------------
# 2. Xử lý giá trị thiếu bằng MICE
# ------------------------------------------------------------------
print("\n" + "=" * 80)
print("DỮ LIỆU TRƯỚC KHI XỬ LÝ GIÁ TRỊ THIẾU")
print(X.head())
print("\nSố giá trị thiếu:")
print(X.isnull().sum())

imputer = IterativeImputer(max_iter=10, random_state=42)

X_imputed = pd.DataFrame(
    imputer.fit_transform(X),
    columns=feature_names
)

print("\n" + "=" * 80)
print("DỮ LIỆU SAU KHI XỬ LÝ GIÁ TRỊ THIẾU (MICE)")
print(X_imputed.head())
print("\nSố giá trị thiếu còn lại:")
print(X_imputed.isnull().sum())

# ------------------------------------------------------------------
# 3. Phát hiện và loại bỏ ngoại lai bằng khoảng cách Euclidean
# ------------------------------------------------------------------
mean_vec = np.mean(X_imputed, axis=0)

dists = np.sqrt(
    np.sum((X_imputed - mean_vec) ** 2, axis=1)
)

threshold = np.percentile(dists, 95)

print("\n" + "=" * 80)
print("THỐNG KÊ NGOẠI LAI")
print(f"Ngưỡng khoảng cách (95%): {threshold:.4f}")
print(f"Số mẫu trước khi loại ngoại lai: {len(X_imputed)}")

mask = dists <= threshold

X_clean = X_imputed[mask]
y_clean = y[mask]

print(f"Số mẫu sau khi loại ngoại lai: {len(X_clean)}")
print(f"Số mẫu bị loại: {len(X_imputed) - len(X_clean)}")

print("\n" + "=" * 80)
print("DỮ LIỆU SAU KHI LOẠI NGOẠI LAI")
print(X_clean.head())

print("\nKích thước cuối cùng:")
print(X_clean.shape)

# Chuyển sang numpy nếu các bước sau cần
X = X_clean.values
y = y_clean.values

print("\n" + "=" * 80)
print(f"✅ Dữ liệu sau tiền xử lý: {X.shape[0]} mẫu, {X.shape[1]} đặc trưng")

DỮ LIỆU GỐC
        ID  Clump_Thickness  Uniformity_Cell_Size  Uniformity_Cell_Shape  \
0  1000025                5                     1                      1   
1  1002945                5                     4                      4   
2  1015425                3                     1                      1   
3  1016277                6                     8                      8   
4  1017023                4                     1                      1   

   Marginal_Adhesion  Single_Epithelial_Cell_Size  Bare_Nuclei  \
0                  1                            2          1.0   
1                  5                            7         10.0   
2                  1                            2          2.0   
3                  1                            3          4.0   
4                  3                            2          1.0   

   Bland_Chromatin  Normal_Nucleoli  Mitoses  Class  
0                3                1        1      2  
1                3        

In [ ]:
!jupyter nbconvert --to html anfis-breast-cancer-model-training.ipynb